In [55]:
import ollama
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

## **Model**

In [56]:
MODEL = "llama3.2:1b"
MAX_STEPS = 5 # acts as a guardrail to how many time sthe model calls the tools
DRAGON_BALL_BASE_URL = "https://dragonball-api.com/"
MOH_URL="https://www.health.go.ke/"

## **Tool**

Composed of three parts: 
- doc string
- type hints eg "city"
- name

In [57]:
def get_weather(city:str) -> str:
    """ Get current weather for a given city"""
    return f"22°C and light rain experienced in {city}."
 
def get_dragon_ball_characters():

    """ 
    Get charactes of the Anime Dragon ball
    """
    url = f"{DRAGON_BALL_BASE_URL}api/characters"
    payload = {}
    headers = {}
    response = requests.request("GET", url, headers=headers, data=payload)
    return response.text
 

def get_latest_articles_by_moh():
    """    Get Ministry of Health in Kenya latest stories, articles and publications  """
    url=f"{MOH_URL}blogs-grid"
    section_selector=".gva-view-grid"
    driver = webdriver.Chrome()
    driver.get(url)
    
    # Wait for the section to load
    section = driver.find_element(By.CSS_SELECTOR, section_selector)
    
    # Parse with BeautifulSoup for easier extraction
    soup = BeautifulSoup(section.get_attribute('innerHTML'), 'html.parser')
    driver.quit()
    
    return soup.get_text(strip=True)

In [58]:
TOOLS = {
    "get_weather":get_weather,
    "get_dragon_ball_characters":get_dragon_ball_characters,  
    "get_latest_articles_by_moh":get_latest_articles_by_moh
}

In [59]:
print(TOOLS)

{'get_weather': <function get_weather at 0x7fb32c1d1440>, 'get_dragon_ball_characters': <function get_dragon_ball_characters at 0x7fb32c1d3d80>, 'get_latest_articles_by_moh': <function get_latest_articles_by_moh at 0x7fb32c1d3a60>}


## **Agent Loop**

COmposed of loop with maximum nuumber of steps, this acts as a guardrail to limit the number of calls made by the model

In [60]:
def run_agent(user_question: str, max_steps=MAX_STEPS):
    # The conversation history. The model has no memory,
    # so we send the whole history on every call.
    messages = [
        {"role": "system", "content": "You can use tools. After you get a tool result, use it to answer the user."},
        {"role": "user", "content": user_question},
    ]

    for step in range(max_steps):
        # 1. Ask the model what to do next
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=  list(TOOLS.values()),
        )

        # 2. Save the model's reply in the history
        messages.append(response.message)

        # 3. No tool requested? Then this is the final answer -> stop
        if not response.message.tool_calls:
            return response.message.content

        # 4. The model asked for tool(s): run each one
        for call in response.message.tool_calls:
            name = call.function.name          # e.g. "get_weather"
            args = call.function.arguments    # e.g. {"city": "Nairobi"}

            # Small models sometimes wrap the real arguments inside "parameters",
            # like a box inside a box. Keep opening until we reach the real ones.
            while "parameters" in args:
                args = args["parameters"]

            print(f"step {step + 1}, model requested: {name}{args}")

            try:
                result = TOOLS[name](**args)   # run the real Python function
            except Exception as e:
                result = f"ERROR: {e}"         # send errors back instead of crashing

            print(f"   tool returned: {result}")

            # 5. Give the tool's result back to the model
            messages.append({"role": "tool", "content": str(result), "tool_name": name})

    return f"Stopped: reached maximum number of steps ({max_steps})"


In [61]:
   # Needs a tool -> watch the loop run.
# print(run_agent("Should I carry an umbrella in Nairobi today?"))
# print("-" * 100)

In [62]:
# print(run_agent("Who are the Dragon Ball Characters?"))
# print("-" * 100)

In [63]:
print(run_agent("What are some of the lates articles by Ministry of Health in Kenya? "))
print("-" * 100)

step 1, model requested: get_latest_articles_by_moh{}
   tool returned: MoH News-09 Nov, 2026Lang’ata Residents Benefit from Free Medical Camp and Government Services OutreachMoH News-06 Nov, 2026Kenya Calls for Stronger African Commitment to Malaria EliminationMoH News-05 Nov, 2026PS Oluga Calls for Stronger Alignment of Development Partner Investments with Health PrioritiesMoH News-05 Nov, 2026Government Sets December 2026 Deadline for Pharmaceutical Manufacturers to Adopt GS1 StandardsMoH News-03 Nov, 2026CS Duale, UNICEF Review Kenya’s Ebola Response and Strengthen Health Emergency PreparednessMoH News-03 Nov, 2026PS Oluga, AFD Discuss Local Pharmaceutical Manufacturing and Digital Health CollaborationMoH News-03 Nov, 2026Kenya Explores Partnership to Expand Access to Essential Antibiotics and Combat Antimicrobial ResistanceMoH News-03 Nov, 2026Ministry of Health and Council of Governors Strengthen Collaboration on Healthcare DeliveryMoH News-02 Nov, 2026Kenya and Beginnings Fund A